In [7]:
import sys
from pathlib import Path

from pyspark.sql import SparkSession

project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "fraud_streaming" / "spark" / "data_processing.py").is_file()
    ),
    Path.cwd(),
)
spark_dir = project_root / "fraud_streaming" / "spark"
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(spark_dir))

from data_processing import compute_window_analytics
from fraud_streaming.redis.config import (
    ANALYTICS_POLL_SECONDS,
    host as redis_host,
    port as redis_port,
)
from fraud_streaming.redis.redis_integration import RedisAnalyticsIntegration

spark = SparkSession.builder.appName("RedisFeatureWindowAnalysis").getOrCreate()
analytics_store = RedisAnalyticsIntegration(host=redis_host, port=redis_port)
analysis_checkpoint_path = (
    project_root / "data" / "checkpoints" / "feature_analysis"
)

In [8]:
def process_feature_analysis_batch(_timer_batch, batch_id):
    transactions = analytics_store.read_feature_store()
    metrics = compute_window_analytics(spark, transactions)
    analytics_store.write_analytics_to_redis(metrics)
    print(f"Updated Redis analytics for batch {batch_id}: {metrics}")

In [9]:
timer_stream = (
    spark.readStream.format("rate")
    .option("rowsPerSecond", 1)
    .load()
)

analysis_query = (
    timer_stream.writeStream
    .foreachBatch(process_feature_analysis_batch)
    .option("checkpointLocation", analysis_checkpoint_path.as_posix())
    .trigger(processingTime=f"{ANALYTICS_POLL_SECONDS} seconds")
    .start()
)
print(f"Started Redis feature analysis; polling every {ANALYTICS_POLL_SECONDS}s")

Started Redis feature analysis; polling every 5s


In [ ]:
try:
    analysis_query.awaitTermination()
finally:
    analysis_query.stop()
    analytics_store.close()

Updated Redis analytics for batch 244: {'transaction_count': 504, 'total_amount': 29190.37, 'average_amount': 57.92, 'category_counts': {'travel': 24, 'kids_pets': 84, 'shopping_net': 42, 'food_dining': 57, 'personal_care': 52, 'health_fitness': 51, 'entertainment': 43, 'home': 85, 'misc_pos': 28, 'shopping_pos': 31, 'misc_net': 5, 'grocery_pos': 2}, 'window_start': '2026-10-04T10:20:16+00:00', 'window_end': '2026-10-04T10:21:16+00:00', 'updated_at': '2026-10-04T10:21:16+00:00'}
Updated Redis analytics for batch 245: {'transaction_count': 624, 'total_amount': 36189.15, 'average_amount': 58.0, 'category_counts': {'travel': 33, 'kids_pets': 103, 'shopping_net': 49, 'food_dining': 65, 'personal_care': 65, 'health_fitness': 62, 'entertainment': 59, 'home': 105, 'misc_pos': 35, 'shopping_pos': 41, 'misc_net': 5, 'grocery_pos': 2}, 'window_start': '2026-10-04T10:20:53+00:00', 'window_end': '2026-10-04T10:21:53+00:00', 'updated_at': '2026-10-04T10:21:53+00:00'}
Updated Redis analytics for bat